# Qualitative Segmentation Examples


In [14]:
from collections import defaultdict
from pathlib import Path
from io import BytesIO
import json
import sys

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import to_rgb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.datasets import DATASETS
from scripts.detectron2_inference import polygons_to_mask, predictions_to_mask

CANDIDATES_PER_CATEGORY = 20  # Change to 10 for smaller galleries.
QUALITY_BOUNDARY = 0.5
DETECTOR_UNSEEN_ONLY = True
PREDICTION_COLUMN = "full__random_forest"
GT_COLOR = "#00bfc4"
PRED_COLOR = "#f28e2b"
CATEGORY_LABELS = {
    "reliable": "Correctly identified high quality",
    "unreliable": "Correctly identified low quality",
    "overconfident": "Estimator overconfidence",
    "underconfident": "Estimator underconfidence",
}
FIG_DIR = ROOT / "paper" / "plots"

SELECTED_IDS = {
    "ca": {"reliable": None, "unreliable": None, "overconfident": None, "underconfident": None},
    "glass": {"reliable": None, "unreliable": None, "overconfident": None, "underconfident": None},
}
EXPORT = False


## Load Saved Data

In [15]:
tables, annotations, detections = {}, {}, {}
for dataset in ("ca", "glass"):
    source = ROOT / "outputs" / "post_detection" / dataset
    targets = pd.read_csv(source / "iou_targets.csv").set_index("row_id").sort_index()
    predictions = pd.read_csv(source / "oof_predictions.csv").set_index("row_id").sort_index()
    assert targets.index.equals(predictions.index)
    assert targets.index.is_unique
    np.testing.assert_allclose(targets["iou_target"], predictions["iou_target"], rtol=1e-6)
    assert targets["fold"].equals(predictions["fold"])
    assert targets["detector_seen_during_training"].equals(predictions["detector_seen_during_training"])
    frame = targets.join(predictions[[PREDICTION_COLUMN]]).rename(columns={PREDICTION_COLUMN: "predicted_iou"})
    assert np.isfinite(frame[["iou_target", "predicted_iou"]].to_numpy()).all()
    frame["error"] = frame["predicted_iou"] - frame["iou_target"]
    frame["abs_error"] = frame["error"].abs()
    true_high = frame["iou_target"] >= QUALITY_BOUNDARY
    predicted_high = frame["predicted_iou"] >= QUALITY_BOUNDARY
    frame["category"] = np.select(
        [true_high & predicted_high, ~true_high & ~predicted_high, ~true_high & predicted_high],
        ["reliable", "unreliable", "overconfident"], default="underconfident",
    )
    if DETECTOR_UNSEEN_ONLY:
        frame = frame.loc[~frame["detector_seen_during_training"]]
    tables[dataset] = frame

    coco = json.loads((ROOT / DATASETS[dataset].dataset_root / "coco_format.json").read_text())
    annotations[dataset] = defaultdict(list)
    for annotation in coco["annotations"]:
        annotations[dataset][annotation["image_id"]].extend(annotation["segmentation"])
    detections[dataset] = {}
    for fold in sorted(frame["fold"].unique()):
        by_image = defaultdict(list)
        for prediction in json.loads((source / f"fold{fold}" / "coco_instances_results.json").read_text()):
            by_image[prediction["image_id"]].append(prediction["segmentation"])
        detections[dataset][int(fold)] = by_image

display(pd.DataFrame({dataset: frame["category"].value_counts() for dataset, frame in tables.items()})
        .reindex(CATEGORY_LABELS).fillna(0).astype(int).rename(index=CATEGORY_LABELS))


,ca,glass
category,,
Correctly identified high quality,345,712
Correctly identified low quality,396,231
Estimator overconfidence,142,114
Estimator underconfidence,95,60


## Sliding Candidate Gallery

In [16]:
def candidate_table(dataset, category):
    frame = tables[dataset].loc[tables[dataset]["category"] == category].reset_index()
    column = {"reliable": "abs_error", "unreliable": "abs_error",
              "overconfident": "error", "underconfident": "error"}[category]
    ascending = category != "overconfident"
    return frame.sort_values([column, "row_id"], ascending=[ascending, True]).head(CANDIDATES_PER_CATEGORY)


def cropped_masks(dataset, row_id):
    record = tables[dataset].loc[row_id]
    image = cv2.imread(str(ROOT / record["image_path"]))
    if image is None:
        raise FileNotFoundError(record["image_path"])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    gt = polygons_to_mask(annotations[dataset][row_id], height, width)
    pred = predictions_to_mask(detections[dataset][int(record["fold"])][row_id], height, width)
    intersection, union = (gt & pred).sum(), (gt | pred).sum()
    actual_iou = intersection / union if union else 1.0
    assert np.isclose(actual_iou, record["iou_target"], rtol=1e-6, atol=1e-7), row_id
    ys, xs = np.where(gt | pred)
    if len(xs):
        padding = max(32, int(0.1 * max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1)))
        x0, x1 = max(0, int(xs.min()) - padding), min(width, int(xs.max()) + padding + 1)
        y0, y1 = max(0, int(ys.min()) - padding), min(height, int(ys.max()) + padding + 1)
    else:
        x0, y0, x1, y1 = 0, 0, width, height
    region = np.s_[y0:y1, x0:x1]
    return image[region], gt[region], pred[region], (x0, y0, x1, y1)


def show_overlay(ax, image, gt, pred):
    overlay = image.astype(np.float32) / 255
    for mask, color in ((gt, GT_COLOR), (pred, PRED_COLOR)):
        rgb = np.array(to_rgb(color))
        overlay[mask] = 0.7 * overlay[mask] + 0.3 * rgb
    ax.imshow(overlay)
    for mask, color in ((gt, GT_COLOR), (pred, PRED_COLOR)):
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            points = contour[:, 0, :]
            if len(points) > 1:
                points = np.vstack([points, points[0]])
                ax.plot(points[:, 0], points[:, 1], color=color, linewidth=0.8)
    ax.axis("off")


def candidate_gallery():
    dataset_picker = widgets.Dropdown(options=[("CA", "ca"), ("Glass", "glass")], description="Dataset:")
    category_picker = widgets.Dropdown(
        options=[(label, name) for name, label in CATEGORY_LABELS.items()],
        description="Example:", layout=widgets.Layout(width="420px"),
    )
    slider = widgets.IntSlider(value=1, min=1, max=1, description="Candidate:", continuous_update=False)
    previous = widgets.Button(description="Previous")
    following = widgets.Button(description="Next")
    choose = widgets.Button(description="Use for paper", button_style="primary")
    details, selection = widgets.HTML(), widgets.HTML()
    picture = widgets.Image(format="png", layout=widgets.Layout(width="100%"))
    candidates = pd.DataFrame()

    def render(change=None):
        previous.disabled = candidates.empty or slider.value == 1
        following.disabled = candidates.empty or slider.value == len(candidates)
        choose.disabled = candidates.empty
        slider.disabled = candidates.empty
        dataset, category = dataset_picker.value, category_picker.value
        selected = SELECTED_IDS[dataset][category]
        selection.value = f"<b>Selected image ID:</b> {selected if selected is not None else 'None yet'}"
        if candidates.empty:
            details.value = "No candidates match this category and filter."
            picture.value = b""
            return
        record = candidates.iloc[slider.value - 1]
        row_id = int(record["row_id"])
        exposure = "detector-seen" if record["detector_seen_during_training"] else "detector-unseen"
        details.value = (
            f"<b>{slider.value}/{len(candidates)} | Image ID {row_id} | fold {int(record['fold'])}</b>"
            f" | Actual IoU {record['iou_target']:.3f}"
            f" | Estimated IoU {record['predicted_iou']:.3f}"
            f" | Error {record['error']:+.3f} | {exposure}"
        )
        image, gt, pred, _ = cropped_masks(dataset, row_id)
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(image)
        axes[0].axis("off")
        axes[0].set_title("Raw crop")
        show_overlay(axes[1], image, gt, pred)
        axes[1].set_title("Ground truth and detector prediction")
        fig.legend(handles=[Patch(color=GT_COLOR, label="Ground truth"), Patch(color=PRED_COLOR, label="Prediction")],
                   loc="lower center", ncol=2, frameon=False)
        fig.tight_layout(rect=(0, 0.07, 1, 1))
        with BytesIO() as stream:
            fig.savefig(stream, format="png", dpi=110, bbox_inches="tight")
            picture.value = stream.getvalue()
        plt.close(fig)
        choose.description = "Selected" if selected == row_id else "Use for paper"
        choose.button_style = "success" if selected == row_id else "primary"

    def refresh(change=None):
        nonlocal candidates
        candidates = candidate_table(dataset_picker.value, category_picker.value)
        slider.value = 1
        slider.max = max(1, len(candidates))
        render()

    def select_current(button):
        record = candidates.iloc[slider.value - 1]
        SELECTED_IDS[dataset_picker.value][category_picker.value] = int(record["row_id"])
        render()

    previous.on_click(lambda button: setattr(slider, "value", slider.value - 1))
    following.on_click(lambda button: setattr(slider, "value", slider.value + 1))
    choose.on_click(select_current)
    slider.observe(render, names="value")
    dataset_picker.observe(refresh, names="value")
    category_picker.observe(refresh, names="value")
    refresh()
    return widgets.VBox([
        widgets.HBox([dataset_picker, category_picker]),
        widgets.HBox([previous, slider, following, choose]),
        details, picture, selection,
    ])


In [17]:
gallery = candidate_gallery()
display(gallery)


## Review Selection And Export

In [18]:
display(pd.DataFrame(SELECTED_IDS).rename(index=CATEGORY_LABELS))
EXPORT = False  # Set True when the chosen examples are ready to export.


,ca,glass
Correctly identified high quality,None,None
Correctly identified low quality,None,None
Estimator overconfidence,None,None
Estimator underconfidence,None,None


In [19]:
for dataset, selected in SELECTED_IDS.items():
    if any(row_id is None for row_id in selected.values()):
        print(f"{dataset.upper()}: choose all four image IDs before preview/export.")
        continue
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    records = []
    for column, (category, row_id) in enumerate(selected.items()):
        record = tables[dataset].loc[row_id]
        assert record["category"] == category, f"ID {row_id} does not belong to {category}"
        image, gt, pred, bounds = cropped_masks(dataset, row_id)
        axes[0, column].imshow(image)
        axes[0, column].axis("off")
        axes[0, column].set_title(f"{CATEGORY_LABELS[category]}\nID {row_id}", fontsize=11)
        show_overlay(axes[1, column], image, gt, pred)
        exposure = "detector-seen" if record["detector_seen_during_training"] else "detector-unseen"
        axes[1, column].set_title(
            f"Actual IoU {record['iou_target']:.3f} | Estimated {record['predicted_iou']:.3f}\n"
            f"Error {record['error']:+.3f} | fold {int(record['fold'])} | {exposure}", fontsize=10)
        records.append({"dataset": dataset, "category": category, "row_id": row_id,
                        "prediction_column": PREDICTION_COLUMN, "quality_boundary": QUALITY_BOUNDARY,
                        **record[["fold", "detector_seen_during_training", "iou_target", "predicted_iou", "error"]].to_dict(),
                        **dict(zip(("crop_x0", "crop_y0", "crop_x1", "crop_y1"), bounds))})
    fig.legend(handles=[Patch(color=GT_COLOR, label="Ground truth"), Patch(color=PRED_COLOR, label="Prediction")],
               loc="lower center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    if EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        label = "CA" if dataset == "ca" else "Glass"
        fig.savefig(FIG_DIR / f"fig_qualitative_{label}.png", dpi=300, bbox_inches="tight")
        pd.DataFrame(records).to_csv(FIG_DIR / f"fig_qualitative_{label}_selection.csv", index=False)
        print(f"Exported {label} figure and selection record to {FIG_DIR}")
    plt.show()
    plt.close(fig)


CA: choose all four image IDs before preview/export.
GLASS: choose all four image IDs before preview/export.
